### This is a basic pulse - measure sequence embedded in a lock-in reference signal


In [ ]:
from spinapi import *
from time import sleep
import pyvisa

# Enable the log file
pb_set_debug(0)

print("Copyright (c) 2015 SpinCore Technologies, Inc.");
print("Using SpinAPI Library version %s" % pb_get_version())
print("Found %d boards in the system.\n" % pb_count_boards())
 
pb_select_board(0)

if pb_init() != 0:
	print("Error initializing board: %s" % pb_get_error())
	input("Please press a key to continue.")
	exit(-1)

# Configure the core clock
pb_core_clock(500)

#Initialize pulse train variables
Tref = 2*ms
Tdelay = 250*us
Tlaser = 100*us
Nrepeats = 1

#---------------------------------------------------------------------------
#
# Write a pulse program 
#
# Define binary strings for each output channel 
#   Note : here we use the first four channels (read right to left) for instrument control
#          and repeat those on channels 5-8 to monitor on an oscilloscope
#
bit0      = 0b00010001
bit1      = 0b00100010
bit2      = 0b01000100
bit3      = 0b10001000
zeros     = 0b00000000

#Program the pulse program -------------------------------------------------
pb_start_programming(PULSE_PROGRAM)

# reference high
loop_high = pb_inst_pbonly(bit0 + bit1, Inst.LOOP, Nrepeats, Tlaser)
# pb_inst_pbonly(bit0, Inst.CONTINUE, 0, Tdelay)
# pb_inst_pbonly(bit0 + bit1, Inst.CONTINUE, 0, Tlaser)
# pb_inst_pbonly(bit0, Inst.END_LOOP, loop_high, Tref/Nrepeats - 2*Tlaser - Tdelay)
pb_inst_pbonly(bit0, Inst.END_LOOP, loop_high, Tref/Nrepeats - Tlaser)
             
# reference low
if Nrepeats > 1:
    loop_low = pb_inst_pbonly(bit1, Inst.LOOP, Nrepeats-1, Tlaser)
    pb_inst_pbonly(zeros, Inst.CONTINUE, 0, Tdelay)
    pb_inst_pbonly(bit1, Inst.CONTINUE, 0, Tlaser)
    pb_inst_pbonly(zeros, Inst.END_LOOP, loop_low, Tref/Nrepeats - 2*Tlaser - Tdelay)
    
pb_inst_pbonly(bit1, Inst.CONTINUE, 0, Tlaser)
pb_inst_pbonly(zeros, Inst.CONTINUE, 0, Tdelay )
pb_inst_pbonly(bit1, Inst.CONTINUE, 0, Tlaser)
pb_inst_pbonly(zeros, Inst.BRANCH, loop_high, Tref/Nrepeats - 2*Tlaser - Tdelay)

pb_stop_programming()
#-----------------------------------------------------------------------------------------

rm = pyvisa.ResourceManager()
#rm.list_resources()
lockin = rm.open_resource('ASRL5::INSTR', baud_rate=9600, data_bits=8, 
                  read_termination = '\r', write_termination = '\r\n')

# Trigger the board
pb_reset() 
pb_start()

print("Pressing a key will stop pulsing\n");
input("Please press a key to stop.")

pb_stop()
pb_close()